# Build RAG Using LlamaIndex (with Google Gemini)

Before running: in Colab open the key icon (Secrets), add `GEMINI_API_KEY`, and turn on notebook access.

### Install the required libraries

In [ ]:
!pip install -U -q llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai pypdf llama-index-readers-file

### Set the Gemini API key

In [ ]:
import os
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")  # stored in Colab Secrets, never paste a key into a cell
assert GEMINI_API_KEY, "Add GEMINI_API_KEY in Colab Secrets and enable notebook access."
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

### Provide the PDF

In [ ]:
PDF_PATH = "/content/the-state-of-ai-how-organizations-are-rewiring-to-capture-value_final.pdf"

if not os.path.exists(PDF_PATH):
    from google.colab import files
    print("PDF not found in /content. Choose the file to upload:")
    uploaded = files.upload()
    PDF_PATH = "/content/" + next(iter(uploaded))

print("Using:", PDF_PATH)

### Build the simple RAG system

In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

# If you hit 503 / overload errors, try another model, for example "gemini-3.7-flash"
LLM_MODEL = "gemini-2.5-flash"
EMBED_MODEL = "gemini-embedding-001"

# Load document
reader = SimpleDirectoryReader(input_files=[PDF_PATH])
documents = reader.load_data()
print(f"Loaded {len(documents)} document(s).")

# Split into chunks
splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)
print(f"Created {len(nodes)} chunks.")

# Set up LLM and embedding model (Gemini)
Settings.llm = GoogleGenAI(model=LLM_MODEL, api_key=GEMINI_API_KEY)
Settings.embed_model = GoogleGenAIEmbedding(
    model_name=EMBED_MODEL,
    api_key=GEMINI_API_KEY,
    embed_batch_size=50,
)

# Create vector index
vector_index = VectorStoreIndex(nodes)

# Create query engine
query_engine = vector_index.as_query_engine(similarity_top_k=3)

### Inspect the vector store

In [ ]:
vector_store = vector_index.vector_store

embedding_dict = vector_store.data.embedding_dict
node_dict = vector_store.data.text_id_to_ref_doc_id

print(f"Number of embeddings: {len(embedding_dict)}")
print(f"Number of node references: {len(node_dict)}")

for i, (node_id, embedding) in enumerate(list(embedding_dict.items())[:3]):
    print(f"\n--- Embedding {i} ---")
    print(f"Node ID: {node_id}")
    print(f"Embedding dimension: {len(embedding)}")
    print(f"First 10 values: {embedding[:10]}")

### Ask questions to the RAG system

In [ ]:
import time

def ask(question, retries=4):
    """Query the RAG system, retrying if Gemini is temporarily overloaded (503) or rate limited (429)."""
    for attempt in range(retries):
        try:
            return query_engine.query(question)
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {repr(e)[:150]}")
            time.sleep(5 * (attempt + 1))
    raise RuntimeError("Gemini kept failing. Wait a minute or change LLM_MODEL and rerun the build cell.")


response = ask("Who is Lareina Yee?")
print(response)

### Check that the responses make sense

In [ ]:
print(len(response.source_nodes))

In [ ]:
print("Source nodes:")
print("=" * 50)

for i, node in enumerate(response.source_nodes):
    print(f"Node {i + 1}:")
    print(f"Score: {node.score}")
    print(f"Text: {node.text[:600]}")
    print(f"Metadata: {node.metadata}")
    print("-" * 30)

In [ ]:
response2 = ask("What are the main findings about AI adoption?")
print(response2)

In [ ]:
response3 = ask("What does the document say about AI risks?")
print(response3)

### Ask your own question

In [ ]:
my_question = "How many organizations use gen AI in at least one business function?"
answer = ask(my_question)
print(answer)
print("\nSources: pages", [n.metadata.get("page_label") for n in answer.source_nodes])